### Environment and Dataset Ingestion

In [1]:
import os
import re
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer

for resource in ["tokenizers/punkt", "tokenizers/punkt_tab", "corpora/stopwords", "corpora/wordnet", "sentiment/vader_lexicon"]:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(resource.split("/")[-1], quiet=True)

RAW_DATA_PATH = os.path.join("..", "dataset", "customer_support_tickets.csv")
CLEANED_DATA_PATH = os.path.join("..", "dataset", "cleaned_customer_support_tickets.csv")

print(f"Loading raw dataset from: {RAW_DATA_PATH}")
df = pd.read_csv(RAW_DATA_PATH)
print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns.")

Loading raw dataset from: ../dataset/customer_support_tickets.csv
Loaded 8469 rows and 17 columns.


---

### Target Feature Engineering — Departmental Routing

**Business Context:**

In an enterprise support desk, incoming complaints are dispatched to functional business units.
So I created a unified ***Target_Department*** by mapping operational ticket subjects into clear functional divisions:
1. Finance & Billing
2. Technical Support
3. Marketing & Sales
4. Account Access & Security
5. Customer Retention & Cancellation


In [2]:
import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Canonical semantic profiles for corporate support units
DEPARTMENT_PROFILES = {
    "Finance & Billing": (
        "billing invoice double charge refund payment transaction fee overcharged bank card disputed deduction price"
    ),
    "Technical Support": (
        "screen flickering device not working hardware crash software bug battery drain power wifi error installation freeze"
    ),
    "Account Access & Security": (
        "forgot password login credentials account locked 2fa security verification sign in access unauthorized profile"
    ),
    "Customer Retention & Cancellation": (
        "cancel subscription terminate plan delete account stop renewal discontinue membership switch competitor end contract"
    ),
    "Marketing & Sales": (
        "product recommendation compatibility compatible discount pricing quote sales enterprise demo feature upgrade"
    )
}

dept_names = list(DEPARTMENT_PROFILES.keys())
print("Defined 5 canonical department profiles.")

/Users/aryanmishra/Documents/ai_app_gisma_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Defined 5 canonical department profiles.


---

### Template Replacement & Weak Supervision Pseudo-Labeling

In [3]:
# 1. Fill placeholders with purchased product
def fill_template_placeholders(row):
    desc = str(row["Ticket Description"])
    product = str(row["Product Purchased"])
    return re.sub(r"\{product_purchased\}", product, desc, flags=re.IGNORECASE)

df["resolved_description"] = df.apply(fill_template_placeholders, axis=1)

# 2. Dense Semantic Labeling using all-MiniLM-L6-v2
print("Loading SBERT encoder for programmatic ground-truth denoising...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Encode department anchor prototypes
dept_vectors = embedder.encode(list(DEPARTMENT_PROFILES.values()), normalize_embeddings=True)

# Encode unique descriptions to minimize computation
unique_descs = df["resolved_description"].unique()
print(f"Encoding {len(unique_descs)} unique descriptions into dense semantic space...")
unique_embeddings = embedder.encode(
    unique_descs.tolist(), 
    batch_size=128, 
    show_progress_bar=True, 
    normalize_embeddings=True
)

# Compute cosine similarity
sim_matrix = cosine_similarity(unique_embeddings, dept_vectors)
best_indices = np.argmax(sim_matrix, axis=1)
best_scores = np.max(sim_matrix, axis=1)

desc_to_label = {desc: dept_names[idx] for desc, idx in zip(unique_descs, best_indices)}
desc_to_confidence = {desc: score for desc, score in zip(unique_descs, best_scores)}

# Map back to dataframe
df["Target_Department"] = df["resolved_description"].map(desc_to_label)
df["Denoise_Confidence"] = df["resolved_description"].map(desc_to_confidence)

# Retain high-confidence, non-ambiguous samples (filters out noisy random text)
CONFIDENCE_THRESHOLD = 0.28
denoised_df = df[df["Denoise_Confidence"] >= CONFIDENCE_THRESHOLD].copy()

print("\n" + "=" * 60)
print(f"Original tickets: {len(df)}")
print(f"Denoised valid tickets retained: {len(denoised_df)} ({len(denoised_df)/len(df)*100:.1f}%)")
print("=" * 60)
print("\nRecovered Ground-Truth Department Distribution:")
display(denoised_df["Target_Department"].value_counts())

Loading SBERT encoder for programmatic ground-truth denoising...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10630.51it/s]


Encoding 8398 unique descriptions into dense semantic space...


Batches: 100%|██████████| 66/66 [00:09<00:00,  6.82it/s]



Original tickets: 8469
Denoised valid tickets retained: 2182 (25.8%)

Recovered Ground-Truth Department Distribution:


Target_Department
Technical Support                    1322
Account Access & Security             541
Finance & Billing                     216
Marketing & Sales                      78
Customer Retention & Cancellation      25
Name: count, dtype: int64

---

### Contraction Expansion, Cleaning & Normalization

In [4]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

for res in ["corpora/stopwords", "corpora/wordnet"]:
    try:
        nltk.data.find(res)
    except LookupError:
        nltk.download(res.split("/")[-1], quiet=True)

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()
filtered_stopwords = stop_words - {"not", "no", "never", "cannot"}

CONTRACTION_MAP = {
    "doesn't": "does not", "don't": "do not", "didn't": "did not",
    "can't": "cannot", "won't": "will not", "isn't": "is not",
    "it's": "it is", "i'm": "i am"
}

def clean_ticket_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    for c, e in CONTRACTION_MAP.items():
        text = text.replace(c, e)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = text.split()
    processed = [
        lemmatizer.lemmatize(w) for w in tokens 
        if w not in filtered_stopwords and len(w) > 1
    ]
    return " ".join(processed)

print("Applying text normalization...")
denoised_df["cleaned_text"] = denoised_df["resolved_description"].apply(clean_ticket_text)

Applying text normalization...


---

### Sentiment & Operational Urgency (SLA Priority)

In [5]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()
CRITICAL_TOKENS = {"not working", "crashes", "data loss", "locked", "error", "broken", "unauthorized", "fail", "failed"}

def assign_sla_priority(text: str):
    scores = sia.polarity_scores(str(text))
    compound = scores["compound"]
    hits = sum(1 for w in CRITICAL_TOKENS if w in str(text).lower())
    adjusted = compound - (hits * 0.15)
    
    if adjusted <= -0.10 or hits >= 2:
        urgency = "Critical"
    elif adjusted < 0.20 or hits >= 1:
        urgency = "High"
    elif adjusted <= 0.50:
        urgency = "Medium"
    else:
        urgency = "Low"
    return pd.Series([compound, urgency])

denoised_df[["sentiment_compound", "Calculated_Priority"]] = denoised_df["resolved_description"].apply(assign_sla_priority)
denoised_df["word_count"] = denoised_df["cleaned_text"].apply(lambda x: len(str(x).split()))

---

### Data Schema Selection & CSV Export

In [6]:
import os

CLEANED_DATA_PATH = os.path.join("..", "dataset", "cleaned_customer_support_tickets.csv")
os.makedirs(os.path.dirname(CLEANED_DATA_PATH), exist_ok=True)

export_columns = [
    "Ticket ID", "Customer Name", "Product Purchased", "Date of Purchase",
    "Target_Department", "Calculated_Priority", "sentiment_compound",
    "word_count", "cleaned_text", "resolved_description"
]

denoised_df[export_columns].to_csv(CLEANED_DATA_PATH, index=False)
print(f"Successfully exported denoised dataset: {CLEANED_DATA_PATH}")
print(f"Final valid rows: {len(denoised_df)}")

Successfully exported denoised dataset: ../dataset/cleaned_customer_support_tickets.csv
Final valid rows: 2182
